In [12]:
import cv2
import numpy as np

# -----------------------------
# Camera Projection Matrices
# -----------------------------

# Camera 1 projection matrix (Identity camera)
P1 = np.array([[1, 0, 0, 0],
               [0, 1, 0, 0],
               [0, 0, 1, 0]], dtype=float)

# Camera 2 projection matrix (shifted camera)
P2 = np.array([[1, 0, 0, -1],
               [0, 1, 0, 0],
               [0, 0, 1, 0]], dtype=float)

# -----------------------------
# Corresponding image points
# -----------------------------

# Point in image 1
pts1 = np.array([[50], [100]], dtype=float)

# Point in image 2
pts2 = np.array([[80], [100]], dtype=float)

# -----------------------------
# Perform Triangulation
# -----------------------------

points_4D = cv2.triangulatePoints(P1, P2, pts1, pts2)

# Convert from homogeneous to 3D coordinates
points_3D = points_4D / points_4D[3]

print("Triangulated 3D Point:")
print("X =", points_3D[0][0])
print("Y =", points_3D[1][0])
print("Z =", points_3D[2][0])

Triangulated 3D Point:
X = -1.6666666666666659
Y = -3.3333333333333326
Z = -0.033333333333333326


In [3]:
import cv2
import numpy as np

# -----------------------------
# Load two images
# -----------------------------
img1 = cv2.imread('image1.jpg')
img2 = cv2.imread('image2.jpg')

gray1 = cv2.cvtColor(img1, cv2.COLOR_BGR2GRAY)
gray2 = cv2.cvtColor(img2, cv2.COLOR_BGR2GRAY)

# -----------------------------
# Detect ORB features
# -----------------------------
orb = cv2.ORB_create(2000)

kp1, des1 = orb.detectAndCompute(gray1, None)
kp2, des2 = orb.detectAndCompute(gray2, None)

# -----------------------------
# Feature Matching
# -----------------------------
bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)
matches = bf.match(des1, des2)

matches = sorted(matches, key=lambda x: x.distance)

# Extract matched keypoints
pts1 = np.float32([kp1[m.queryIdx].pt for m in matches]).reshape(-1,1,2)
pts2 = np.float32([kp2[m.trainIdx].pt for m in matches]).reshape(-1,1,2)

# -----------------------------
# Camera Intrinsic Matrix
# (Example values)
# -----------------------------
K = np.array([[800, 0, 320],
              [0, 800, 240],
              [0, 0, 1]])

# -----------------------------
# Estimate Essential Matrix
# -----------------------------
E, mask = cv2.findEssentialMat(pts1, pts2, K, method=cv2.RANSAC)

# -----------------------------
# Recover Camera Motion
# -----------------------------
_, R, t, mask = cv2.recoverPose(E, pts1, pts2, K)

print("Rotation Matrix:\n", R)
print("\nTranslation Vector:\n", t)

# -----------------------------
# Projection Matrices
# -----------------------------
P1 = np.hstack((np.eye(3), np.zeros((3,1))))
P2 = np.hstack((R, t))

P1 = K @ P1
P2 = K @ P2

# -----------------------------
# Triangulate Points
# -----------------------------
points_4D = cv2.triangulatePoints(P1, P2, pts1[:20].T, pts2[:20].T)

# Homogeneous coordinate
w = points_4D[3]

# Avoid division by zero
valid_points = w != 0

points_3D = points_4D[:, valid_points] / w[valid_points]

# Keep only X,Y,Z
points_3D = points_3D[:3]

print("3D Points:")
print(points_3D)

Rotation Matrix:
 [[ 0.0011904  -0.99239464 -0.12309129]
 [-0.99881069  0.00481979 -0.0485178 ]
 [ 0.04874208  0.12300265 -0.99120863]]

Translation Vector:
 [[0.07388255]
 [0.02367702]
 [0.99698584]]
3D Points:
[[ 3.6902607e-03 -1.1921604e+00 -0.0000000e+00 -0.0000000e+00
  -0.0000000e+00 -0.0000000e+00 -0.0000000e+00 -0.0000000e+00
  -0.0000000e+00 -0.0000000e+00 -0.0000000e+00]
 [-1.3003891e-10 -1.2095897e+00 -1.9154143e-38 -1.1889309e-02
   1.1419414e-38 -0.0000000e+00 -0.0000000e+00 -0.0000000e+00
  -1.2708478e-38  8.1262808e-35  1.7801179e-38]
 [-1.8272161e-39 -1.7669836e-02 -0.0000000e+00 -0.0000000e+00
  -9.9500378e-31 -1.1146399e-38  1.4012985e-45 -2.0317845e-38
   0.0000000e+00  1.1053633e-38  1.5233682e-07]]


In [9]:
import cv2
import numpy as np

img1 = cv2.imread('image1.jpg',0)
img2 = cv2.imread('image2.jpg',0)

orb = cv2.ORB_create(2000)

kp1, des1 = orb.detectAndCompute(img1,None)
kp2, des2 = orb.detectAndCompute(img2,None)
bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)
matches = bf.match(des1, des2)

matches = sorted(matches, key=lambda x:x.distance)
pts1 = np.float32([kp1[m.queryIdx].pt for m in matches])
pts2 = np.float32([kp2[m.trainIdx].pt for m in matches])
F, mask = cv2.findFundamentalMat(pts1, pts2, cv2.FM_RANSAC)

print("Fundamental Matrix:\n", F)
K = np.array([[800,0,320],
              [0,800,240],
              [0,0,1]])

E = K.T @ F @ K

print("Essential Matrix:\n", E)
_, R, t, mask = cv2.recoverPose(E, pts1, pts2, K)

print("Rotation:\n", R)
print("Translation:\n", t)
P1 = np.hstack((np.eye(3), np.zeros((3,1))))
P2 = np.hstack((R,t))

P1 = K @ P1
P2 = K @ P2

pts1 = pts1.T
pts2 = pts2.T

points_4D = cv2.triangulatePoints(P1, P2, pts1[:,:20], pts2[:,:20])

points_3D = points_4D / points_4D[3]

print(points_3D[:3])

Fundamental Matrix:
 [[-3.93328994e-08 -4.17186737e-07  7.39456510e-05]
 [ 8.96112583e-07  1.87345633e-05 -3.52180382e-03]
 [-1.60580349e-04 -5.21059110e-03  1.00000000e+00]]
Essential Matrix:
 [[-2.51730556e-02 -2.66999512e-01 -3.10125549e-02]
 [ 5.73512053e-01  1.19901205e+01  1.00899791e+00]
 [ 3.35201147e-02 -6.78236539e-01 -1.16332232e-02]]
Rotation:
 [[ 0.56632602 -0.14584335 -0.8111748 ]
 [-0.80026906  0.13805298 -0.58353304]
 [ 0.19708951  0.97962804 -0.03853092]]
Translation:
 [[0.98226652]
 [0.03232024]
 [0.18468321]]
[[-0.29108763 -0.28746107 -0.24987233 -0.255342   -0.29113114 -0.29141247
  -0.25515234 -0.25599107 -0.2615008  -0.2643998  -0.28729588]
 [ 0.01761158  0.02312585  0.09336985  0.03556949  0.0654139   0.066366
   0.0807715   0.03266894  0.07057306  0.01383503  0.07044316]
 [ 0.8939645   0.88521653  0.890224    0.87965924  0.8903037   0.89026445
   0.87292296  0.8804617   0.88621795  0.88045007  0.8785287 ]]
